In [13]:
import pandas as pd
import os
import subprocess

In [14]:
# Get repo root and set folders
root = subprocess.run(["git", "rev-parse", "--show-toplevel"], capture_output=True, text=True).stdout.strip()
data_folder = os.path.join(root, "GBI Optimisation", "data")
output_folder = os.path.join(root, "GBI Optimisation")

# Get excel file and sheets
excel_path = os.path.join(data_folder, "salaryP1 Nominal.xlsx")

# Load your salary data
salary_data = pd.read_excel(excel_path)  # Adjust path if needed

# Rename columns for clarity
salary_data.columns = ['Year', 'Gross_Salary_DKK']

In [15]:
# Constants
inflation_rate = 0.02
nominal_investment_return = 0.084
real_investment_return = nominal_investment_return - inflation_rate
tax_on_gains = 0.153
ratepension_contrib_own = 0.05
ratepension_contrib_company = 0.10
ratepension_total_contrib = ratepension_contrib_own + ratepension_contrib_company
initial_ratepension_yearly_cost = 888  # DKK
aldersopsparing_contrib = 9400  # DKK per year
avg_post_tax_rate = 0.63  # average net salary percentage after tax


In [16]:

# Initialize columns
salary_data['Ratepension_Contribution'] = salary_data['Gross_Salary_DKK'] * ratepension_total_contrib
salary_data['Aldersopsparing_Contribution'] = aldersopsparing_contrib
salary_data['Ratepension_Balance'] = 0.0
salary_data['Aldersopsparing_Balance'] = 0.0

# Build balances
ratepension_balance = 0
aldersopsparing_balance = 0
ratepension_yearly_cost = initial_ratepension_yearly_cost

for idx, row in salary_data.iterrows():
    # Contributions
    ratepension_contribution = row['Ratepension_Contribution']
    aldersopsparing_contribution = row['Aldersopsparing_Contribution'] / avg_post_tax_rate  # gross up aldersopsparing

    # Apply investment returns after tax and deduct cost
    if idx > 0:
        ratepension_balance = (ratepension_balance - ratepension_yearly_cost) * (1 + real_investment_return * (1 - tax_on_gains))
        aldersopsparing_balance = aldersopsparing_balance * (1 + real_investment_return * (1 - tax_on_gains))

        # Inflate administration cost yearly
        ratepension_yearly_cost *= (1 + inflation_rate)

    # Add yearly contributions
    ratepension_balance += ratepension_contribution
    aldersopsparing_balance += aldersopsparing_contribution

    # Update balances in DataFrame
    salary_data.at[idx, 'Ratepension_Balance'] = ratepension_balance
    salary_data.at[idx, 'Aldersopsparing_Balance'] = aldersopsparing_balance

# Retirement phase (after accumulation)
retirement_start_year = salary_data['Year'].max() + 1
retirement_years = 25

# Last known balances
final_ratepension_balance = ratepension_balance
final_aldersopsparing_balance = aldersopsparing_balance

# Retirement withdrawals
ratepension_yearly_income = final_ratepension_balance / retirement_years
aldersopsparing_lump_sum = final_aldersopsparing_balance  # taken at first year of retirement

# Build retirement DataFrame
retirement_data = {
    'Year': [],
    'Ratepension_Income': [],
    'Aldersopsparing_Withdrawal': []
}

for i in range(retirement_years):
    year = retirement_start_year + i
    ratepension_income = ratepension_yearly_income * ((1 + inflation_rate) ** i)
    aldersopsparing_withdrawal = aldersopsparing_lump_sum if i == 0 else 0

    retirement_data['Year'].append(year)
    retirement_data['Ratepension_Income'].append(ratepension_income)
    retirement_data['Aldersopsparing_Withdrawal'].append(aldersopsparing_withdrawal)

retirement_df = pd.DataFrame(retirement_data)


In [17]:
print(salary_data)

    Year  Gross_Salary_DKK  Ratepension_Contribution  \
0   2025      43871.000000               6580.650000   
1   2026      45187.130000               6778.069500   
2   2027      49253.971700               7388.095755   
3   2028      50731.590851               7609.738628   
4   2029      52253.538577               7838.030786   
5   2030      53821.144734               8073.171710   
6   2031      58665.047760               8799.757164   
7   2032      60424.999193               9063.749879   
8   2033      62237.749168               9335.662375   
9   2034      64104.881643               9615.732247   
10  2035      69874.320991              10481.148149   
11  2036      71970.550621              10795.582593   
12  2037      74129.667140              11119.450071   
13  2038      76353.557154              11453.033573   
14  2039      83225.377298              12483.806595   
15  2040      85722.138617              12858.320793   
16  2041      88293.802775              13244.07

In [18]:
print(retirement_df)

    Year  Ratepension_Income  Aldersopsparing_Withdrawal
0   2070        85167.499280                2.605229e+06
1   2071        86870.849265                0.000000e+00
2   2072        88608.266251                0.000000e+00
3   2073        90380.431576                0.000000e+00
4   2074        92188.040207                0.000000e+00
5   2075        94031.801011                0.000000e+00
6   2076        95912.437032                0.000000e+00
7   2077        97830.685772                0.000000e+00
8   2078        99787.299488                0.000000e+00
9   2079       101783.045477                0.000000e+00
10  2080       103818.706387                0.000000e+00
11  2081       105895.080515                0.000000e+00
12  2082       108012.982125                0.000000e+00
13  2083       110173.241767                0.000000e+00
14  2084       112376.706603                0.000000e+00
15  2085       114624.240735                0.000000e+00
16  2086       116916.725550   

In [19]:
salary_data.to_csv(os.path.join(output_folder, "pension_contributionP1.csv"))

In [20]:
retirement_df.to_csv(os.path.join(output_folder, "retirement_infoP1.csv"))